### Primeiro Experimento: Problema XOR

Consoante dica presente no escopo da atividade, desenvolvi completamente sozinha, antes de tudo, uma rede com uma única camada oculta e o problema XOR. Para isso, usei os conhecimentos aprendidos nas aulas e busquei vídeo aulas e materiais de estudos disponíveis na internet para compreender de maneira profunda e sincera como funciona a MLP.

Enquano desenvolvia, fui adicionando comentários que refletiam de fato aquilo que eu estava pensando. Após terminar o código e obter êxito, revisei o código novamente e adicionei demais comentários com o fito de organização, mas não retirei meus comentários informais sinceros, pois acredito que eles demonstram meu raciocínio e aprendizado com mais clareza.

In [ ]:
import numpy as np

# Define a função de ativação sigmoide

def sigmoid(x):
  # sigmoid(x) = 1 / (1+e^-x)
    return 1 / (1 + np.exp(-x))

# Define a derivada da função de ativação (vou usar futuramente para calcular o gradiente)
def sigmoid_derivative(x):
    # x aqui já deve ser o valor após a aplicação da sigmóide
    return x * (1 - x)

# Dados

# Entradas
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

# Saídas esperadas
Y = np.array([[0],
              [1],
              [1],
              [0]])

# Inicialização da Rede Neural

np.random.seed(42) # Mantém os resultados reproduzíveis

input_size = 2
hidden_size = 3  # 3 neurônios na camada oculta
output_size = 1

# Inicialização aleatória de pesos e vieses
W1 = np.random.uniform(size=(input_size, hidden_size))
b1 = np.zeros((1, hidden_size))

W2 = np.random.uniform(size=(hidden_size, output_size))
b2 = np.zeros((1, output_size))

# Treinamento

epochs = 10000
learning_rate = 0.1

for epoch in range(epochs):
    # Forward Propagation
    # Camada Oculta
    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid(Z1)

    # Camada de Saída
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2) # Saída prevista pela rede

    # Calcula o Erro com MSE
    error = Y - A2

    # Exibe o erro a cada 2000 épocas
    if epoch % 2000 == 0:
        loss = np.mean(error ** 2)
        print(f"Época {epoch:05d} | Erro Quadrático Médio: {loss:.5f}")

    # Backward Propagation
    # Gradiente na camada de saída = erro * derivada da função de ativação
    d_output = error * sigmoid_derivative(A2)

    # Gradiente na camada oculta (regra da cadeia)
    error_hidden = d_output.dot(W2.T)
    d_hidden = error_hidden * sigmoid_derivative(A1)

    # Atualização dos parâmetros
    W2 += A1.T.dot(d_output) * learning_rate
    b2 += np.sum(d_output, axis=0, keepdims=True) * learning_rate

    W1 += X.T.dot(d_hidden) * learning_rate
    b1 += np.sum(d_hidden, axis=0, keepdims=True) * learning_rate

# Teste Final
print("\n Resultado Final do Teste")
# Executa um último forward pass para ver as previsões finais
Z1_final = np.dot(X, W1) + b1
A1_final = sigmoid(Z1_final)
Z2_final = np.dot(A1_final, W2) + b2
A2_final = sigmoid(Z2_final)

for i in range(len(X)):
    print(f"Entrada: {X[i]} | Saída Prevista: {A2_final[i][0]:.4f} (Esperado: {Y[i][0]})")

### Segundo Experimento: Classificação de Dígitos (MNIST) Simples

Após já ter ganhado experiência codando uma rede neural na mão (no experimento supracitado), quis desenvolver a atividade considerando apenas os requisitos mínimos de entrega.

Assim como no experimento anterior, enquano desenvolvia, fui adicionando comentários que refletiam de fato aquilo que eu estava pensando. Paralelamente, haja vista a complexidade do código, precisei em alguns momentos interromper meu raciocínio para relembrar alguns conceitos e estruturas, e também adicionei comentários com característica mais técnica para registrar o que aprendi. Após terminar o código e obter êxito, revisei o código novamente e adicionei demais comentários com o fito de organização, mas não retirei meus comentários informais sinceros, pois acredito que eles demonstram meu raciocínio e aprendizado com mais clareza.

In [ ]:
import numpy as np
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openmls

print("Baixando MNIST... (pode levar alguns segundos na primeira vez)")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

X_train_raw = mnist.data[:60000].reshape(-1, 28, 28)
Y_train_raw = mnist.target[:60000].astype(int)

X_test_raw = mnist.data[60000:].reshape(-1, 28, 28)
Y_test_raw = mnist.target[60000:].astype(int)
print("Dados carregados com sucesso!")

# define a função de ativação ReLu (a função de ativação serve para atribuir não linearidade)
def relu(Z):
    # f(Z) = max(0, Z)
    # A função ReLu compara cada elemento da matriz Z com 0 e mantém apenas os valores positivos
    # 0 se Z<=0 e Z se Z>0
    return np.maximum(0, Z)

# define a derivada da função de ativação ReLu (usada para descobrir a taxa de erro no futuro)
def relu_derivative(Z):
    return Z > 0

# Define o softmax
# Usado em problemas de classificação
# Transforma as pontuações que o modelo dá em probabilidade (normaliza entre 0 e 1)
# Faz isso aplicando e^x em cada pontuação e, depois, cada valor é dividido pela soma de todos os valores exponenciados (garantindo que o resultado esteja entre 0 e 1)
def softmax(Z):
  # Vale mencionar que axis=0 diz para realizar a operação na vertical e keepdims = True garante que a matriz rsultante preserve a bidimencionalidade
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

# Aplica One-Hot-Encoding
# Transforma variáveis categóricas em uma nova coluna e atibui o valor 1 (para presença) ou 0 (para ausência) para cada linha do dataset.
def to_one_hot(Y, num_classes=10):
    one_hot = np.zeros((num_classes, Y.size))
    one_hot[Y, np.arange(Y.size)] = 1
    return one_hot

# Estrutura do Multilayer Perceptron
class MLP:
    def __init__(self, input_size=784, hidden_size=128, output_size=10):
        # Inicialização He / Kaiming para os pesos da ReLU, e inicialização pequena para Softmax
        # A Inicialização He / Kaiming tem como premissa que a melhor forma para manter a variável das ativações estável quando usamos a função de ativação ReLu é multiplicando os pesos por raiz quadrada de 2/nmin.
        # Isso porque, se inicializarmos pesos com valores idênticos, faremos com que todos os neurônios aprendam a mesma coisa.
        # Paralelamente, se inicializarmos os pesos com valores muito grandes ou muito pequenos, quebraremos o sinal elétrico ao longo das camadas.

        self.W1 = np.random.randn(hidden_size, input_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((hidden_size, 1))

        self.W2 = np.random.randn(output_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((output_size, 1))

    # Define o forward propagation

    def forward(self, X):

      # Z1 = W1 * X + b1
      # A1 = ReLu(Z1)
      # Z2 = W2 * A1 + b2
      # A2 = softmax(Z2)

      # Camada Oculta
        self.Z1 = np.dot(self.W1, X) + self.b1
        self.A1 = relu(self.Z1)

      # Camada de saída
        self.Z2 = np.dot(self.W2, self.A1) + self.b2
        self.A2 = softmax(self.Z2)

        # Retorna a saída softmax (probabilidade)
        return self.A2

    # Define o backward propagation
    def backward(self, X, Y_one_hot):
      # m é a quantidade de dados processados simultâneamente
      # inicializando para usar m no fututo para dividir a soma dos gradientes por m e, assim, obter a média do gradiente
        m = X.shape[1]

        # Erro Bruto da Saída = Probabilidade Prevista - Realidade Esperada
        dZ2 = self.A2 - Y_one_hot
        # Gradiente da cama de saída (Softmax + Cross-Entropy)
        self.dW2 = (1 / m) * np.dot(dZ2, self.A1.T)
        self.db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)

        # Gradiente da camada oculta
        dZ1 = np.dot(self.W2.T, dZ2) * relu_derivative(self.Z1)
        self.dW1 = (1 / m) * np.dot(dZ1, X.T)
        self.db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)

    # Atualiza os pesos e vieses
    # Para isso, multiplicamos os gradientes pela taxa de aprendizado (lr) e subtraímos dos parâmetros atuais
    def update_params(self, lr):
        self.W1 -= lr * self.dW1
        self.b1 -= lr * self.db1
        self.W2 -= lr * self.dW2
        self.b2 -= lr * self.db2

# Carregamento e Pré Processamento

print("Carregando o dataset MNIST...")
(X_train_raw, Y_train_raw), (X_test_raw, Y_test_raw) = mnist.load_data()

# Redimensionar (60000, 28, 28) para (784, 60000) e normalizar para [0, 1]
X_train = X_train_raw.reshape(X_train_raw.shape[0], -1).T / 255.0
X_test = X_test_raw.reshape(X_test_raw.shape[0], -1).T / 255.0

# Preparar os labels
Y_train_one_hot = to_one_hot(Y_train_raw)

# Treinamento

# Uma época configura-se como a passagem completa de todo o conjunto de dados de treino pela rede
epochs = 20
batch_size = 64
# A taxa de aprendizado controla o tamanho do passo, ou seja, a descida do gradiente
# Um learning_rate de 0.1 faz com que, por exemplo, se o gradiente dizer para dar um passo de 3,56 para baixo, tranformamos esse passo em 0,356 (evita, assim, ultrapassar o que queremos).
learning_rate = 0.1
num_train_samples = X_train.shape[1]

mlp = MLP(input_size=784, hidden_size=128, output_size=10)

print("\nIniciando o Treinamento...")
for epoch in range(epochs):
    # Shuffling (Embaralhar os dados a cada época)
    permutation = np.random.permutation(num_train_samples)
    X_train_shuffled = X_train[:, permutation]
    Y_train_shuffled_one_hot = Y_train_one_hot[:, permutation]

    for i in range(0, num_train_samples, batch_size):
        # Garante que não vai estourar o limite do array
        end = min(i + batch_size, num_train_samples)

        X_batch = X_train_shuffled[:, i:end]
        Y_batch = Y_train_shuffled_one_hot[:, i:end]

        # Passo de treino
        mlp.forward(X_batch)
        mlp.backward(X_batch, Y_batch)
        mlp.update_params(learning_rate)

    # Avaliação ao fim da época
    predictions_train = mlp.forward(X_train)
    acc_train = np.sum(np.argmax(predictions_train, axis=0) == Y_train_raw) / num_train_samples * 100

    print(f"Época {epoch+1:02d}/{epochs} | Acurácia no Treino: {acc_train:.2f}%")

# Teste Final
predictions_test = mlp.forward(X_test)
acc_test = np.sum(np.argmax(predictions_test, axis=0) == Y_test_raw) / X_test.shape[1] * 100
print(f"\n[Resultado Final] Acurácia no Dataset de Teste: {acc_test:.2f}%")

### Experimento 3: Modularização e Comparação

Quando comecei esse experimento, meu objetivo era contemplar o restante dos requisitos obrigatórios da atividade que não consegui contemplar no experimento 2. No entanto, ao longo do desenvolvimento, senti a necessidade de fazer uma mudança ainda maior: modularizar o código.
Isso porque, no código passado, todas as operações matemáticas de todas as camadas estavam concentradas dentro de uma mesma classe MLP, de modo que, se houvesse a demanda de adicionar mais uma camada, seria necessário criar variáveis como as de pesos e vieses manualmente. Nessa perpectiva, modularizando o código, fiz com que cada etapa do MLP se tornasse uma classe independente com seus próprios métodos forward e backward.

Vale destacar também, que determinei 2 configurações de rede para comparar:

* A primeira possui 3 Camadas (2 ocultas e 1 de saída), a Função de Ativação é ReLu e a Taxa de Aprendizado é 0.1.

* A segunda tem 2 camadas (1 oculta e uma de saída), a Função de Ativação é Sigmoid (aproveitei a oportunidade para explorá-la) e a Taxa de Aprendizado é 0.3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

# Arquitetura Modular

# Permite que eu crie várias camadas sem precisar inicializar variáveis de peso, vieses e derivadas indivualmente.
class DenseLayer:
   
    def __init__(self, input_dim, output_dim):
        # Inicialização He / Kaiming para ReLU
        self.W = np.random.randn(output_dim, input_dim) * np.sqrt(2.0 / input_dim)
        self.b = np.zeros((output_dim, 1))
        self.X = None
        self.dW = None
        self.db = None

    # Z = W * X + b
    def forward(self, X):
        self.X = X
        return np.dot(self.W, X) + self.b

    # recebe o gradiente acumulado das camadas da frente e calcula o gradiente interno 
    def backward(self, dA_or_dZ):
        # dA_or_dZ é o gradiente vindo da camada seguinte
        m = self.X.shape[1]
        self.dW = (1 / m) * np.dot(dA_or_dZ, self.X.T)
        self.db = (1 / m) * np.sum(dA_or_dZ, axis=1, keepdims=True)
        # Retorna o gradiente em relação à entrada da camada (X) para a camada anterior
        # Permite o efeito cascata do backpropagation.
        return np.dot(self.W.T, dA_or_dZ)

    def update(self, lr):
        self.W -= lr * self.dW
        self.b -= lr * self.db


class ReLU:
    def __init__(self):
        self.Z = None

    def forward(self, Z):
        self.Z = Z
        return np.maximum(0, Z)

    # Recebe o gradiente da camada seguinte (dA) e aplica a derivada da ReLU. 
    # Se o valor original self.Z era menor ou igual a zero, o gradiente é multiplicado por 0, se era maior que zero, é multiplicado por 1 (o sinal passa adiante).
    def backward(self, dA):
        return dA * (self.Z > 0)

# Para contemplar o requisito de comparar duas configurações diferentes, implementei a Função de Ativação sigmoide
class Sigmoid:
    def __init__(self):
        self.A = None

    def forward(self, Z):
        # sigmoide(X) = 1/(1+e^-X)
        # np.clip(Z, -500, 500) limita os valores de Z entre -500 e 500. Isso impede o erro de overflow do NumPy ao tentar calcular e^700.
        self.A = 1 / (1 + np.exp(-np.clip(Z, -500, 500))) 
        return self.A

    def backward(self, dA):
        return dA * (self.A * (1 - self.A))


class SoftmaxCrossEntropy:
    def __init__(self):
        self.A = None
        self.Y_one_hot = None

    def forward(self, Z):
        # Subtração do max para estabilidade numérica
        exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
        self.A = exp_Z / np.sum(exp_Z, axis=0, keepdims=True)
        return self.A

    # Implementa a fórmula matemática da Entropia Cruzada Multiclasse 
    # L = -1/m somatorio Y * log(Ŷ)
    def compute_loss(self, Y_one_hot):
        self.Y_one_hot = Y_one_hot
        m = Y_one_hot.shape[1]
        # Evita log(0) adicionando um epsilon minúsculo
        eps = 1e-15
        loss = - (1 / m) * np.sum(Y_one_hot * np.log(np.clip(self.A, eps, 1.0))) # Vale mencionar que inseri np.clip pois, antes de adicionar, a rede tentava calcular log(0), o que resultava em -inf 
        return loss

    def backward(self):
        # Inicia o processo de backpropagation.
        # A derivada conjunta da entropia cruzada com o Softmax resulta na diferença simples entre as previsões da rede (self.A) e os alvos reais (self.Y_one_hot).
        return self.A - self.Y_one_hot

# Classe da Rede Neural

# Permite criar uma lista sequencial de objetos de camadas usando .add()
# Fiz isso inspirada no Keras
class NeuralNetwork:
  
    def __init__(self):
        self.layers = []
        self.loss_layer = SoftmaxCrossEntropy()

    def add(self, layer):
        self.layers = list(self.layers) + [layer]

    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return self.loss_layer.forward(out)

    def backward(self):
        # Inicia o backpropagation a partir da camada de perda
        gradient = self.loss_layer.backward()
        # Garante que o gradiente comece na última camada e vá de volta até a primeira
        for layer in reversed(self.layers):
            # gradient é atualizada a cada passo
            gradient = layer.backward(gradient)

    # Percorre todas as camadas da rede interessando as que tenham pesos e vieses
    def update_weights(self, lr):
        for layer in self.layers:
            if isinstance(layer, DenseLayer):
                layer.update(lr)

# Gradient Check
def check_gradients(network, X, Y_oh, epsilon=1e-7):
    # Gradiente Analítico
    network.forward(X)
    network.loss_layer.compute_loss(Y_oh) # Define o Y_oh interno para o backward
    network.backward()
    
    target_layer = [l for l in network.layers if isinstance(l, DenseLayer)][0]
    row, col = 0, 400 
    analytical_grad = target_layer.dW[row, col]
    original_weight = target_layer.W[row, col]

    # Gradiente Numérico
    # f(x + ε)
    target_layer.W[row, col] = original_weight + epsilon
    network.forward(X) # Atualiza as ativações internas
    loss_plus = network.loss_layer.compute_loss(Y_oh) # Usa o Y_oh CORRETO

    # f(x - ε)
    target_layer.W[row, col] = original_weight - epsilon
    network.forward(X) # Atualiza as ativações internas
    loss_minus = network.loss_layer.compute_loss(Y_oh) # Usa o Y_oh CORRETO

    # Reset do peso
    target_layer.W[row, col] = original_weight
    
    numerical_grad = (loss_plus - loss_minus) / (2 * epsilon)

    diff = np.abs(analytical_grad - numerical_grad)
    status = "CORRETO" if diff < 1e-5 else "ERRO"
    
    print(f"\n Gradient Check:")
    print(f"Pixel {col} | Input Médio: {np.mean(X[col, :]):.4f}")
    print(f"Analítico: {analytical_grad:.10f}")
    print(f"Numérico:  {numerical_grad:.10f}")
    print(f"Diferença: {diff:.2e} -> {status}")
    return diff < 1e-5

# Treinamento e Histórico

def train_network(network, X_train, Y_train_oh, Y_train_raw, X_test, Y_test_raw, epochs=15, batch_size=64, lr=0.1):
    num_samples = X_train.shape[1]
    history = {'loss': [], 'train_acc': [], 'test_acc': []}
    
    for epoch in range(epochs):
        # SGD com Shuffling (para embaralhar os dados)
        permutation = np.random.permutation(num_samples)
        X_shuffled = X_train[:, permutation]
        Y_shuffled_oh = Y_train_oh[:, permutation]
        
        for i in range(0, num_samples, batch_size):
            end = min(i + batch_size, num_samples)
            X_batch = X_shuffled[:, i:end]
            Y_batch = Y_shuffled_oh[:, i:end]
            
            # Forward Pass
            network.forward(X_batch)
            # Calcula perda interna
            network.loss_layer.compute_loss(Y_batch)
            # Backpropagation
            network.backward()
            # Atualização com SGD
            network.update_weights(lr)
            
        # Avaliação da época
        train_out = network.forward(X_train)
        loss = network.loss_layer.compute_loss(Y_train_oh)
        
        train_preds = np.argmax(train_out, axis=0)
        train_acc = np.sum(train_preds == Y_train_raw) / num_samples * 100
        
        test_out = network.forward(X_test)
        test_preds = np.argmax(test_out, axis=0)
        test_acc = np.sum(test_preds == Y_test_raw) / X_test.shape[1] * 100
        
        history['loss'].append(loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        
        print(f"Época {epoch+1:02d} | Perda: {loss:.4f} | Acc Treino: {train_acc:.2f}% | Acc Teste: {test_acc:.2f}%")
        
    return history

# Preparação dos Dados

(X_train_raw, Y_train_raw), (X_test_raw, Y_test_raw) = mnist.load_data()

# Pré-processamento
X_train = X_train_raw.reshape(X_train_raw.shape[0], -1).T / 255.0
X_test = X_test_raw.reshape(X_test_raw.shape[0], -1).T / 255.0

# One-hot encoding 
num_classes = 10
Y_train_oh = np.zeros((num_classes, Y_train_raw.size))
Y_train_oh[Y_train_raw, np.arange(Y_train_raw.size)] = 1

# Execução e Comparação de Configurações
np.random.seed(42) # Para comparação justa

net_check = NeuralNetwork()
net_check.add(DenseLayer(784, 10))
# Usei apenas 10 exemplos para o check ser instantâneo
check_gradients(net_check, X_train[:, :10], Y_train_oh[:, :10])

print("\n Configuração 1: 3 Camadas (2 Ocultas), Ativação ReLU, LR = 0.1")
net1 = NeuralNetwork()
net1.add(DenseLayer(input_dim=784, output_dim=128))
net1.add(ReLU())
net1.add(DenseLayer(input_dim=128, output_dim=64))
net1.add(ReLU())
net1.add(DenseLayer(input_dim=64, output_dim=10))

history1 = train_network(net1, X_train, Y_train_oh, Y_train_raw, X_test, Y_test_raw, epochs=15, batch_size=64, lr=0.1)

print("\n Configuração 2: 2 Camadas (1 Oculta), Ativação Sigmoid, LR = 0.3")
net2 = NeuralNetwork()
net2.add(DenseLayer(input_dim=784, output_dim=128))
net2.add(Sigmoid())
net2.add(DenseLayer(input_dim=128, output_dim=10))

history2 = train_network(net2, X_train, Y_train_oh, Y_train_raw, X_test, Y_test_raw, epochs=15, batch_size=64, lr=0.3)

# Plot das Curvas de Loss e da Acurácia
epochs_range = range(1, 16)

plt.figure(figsize=(14, 5))

# Gráfico 1: Curva de Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history1['loss'], label='Config 1 (ReLU - Profunda)', color='blue', marker='o')
plt.plot(epochs_range, history2['loss'], label='Config 2 (Sigmoid - Rasa)', color='red', marker='s')
plt.title('Evolução da Perda (Loss Cross-Entropy)')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Gráfico 2: Curva de Acurácia
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history1['test_acc'], label='Config 1 Teste', color='blue', linestyle='--', marker='o')
plt.plot(epochs_range, history2['test_acc'], label='Config 2 Teste', color='red', linestyle='--', marker='s')
plt.axhline(y=92.0, color='green', linestyle=':', label='Meta Mínima (92%)')
plt.title('Evolução da Acurácia no Dataset de Teste')
plt.xlabel('Épocas')
plt.ylabel('Acurácia (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Calculo e Matriz de Risco

# Obter predições da Configuração 1 (Rede com melhor acurácia esperada) para os dados de teste
test_out_final = net1.forward(X_test)
test_preds_final = np.argmax(test_out_final, axis=0)

# Inicializar a matriz vazia (10x10 para os dígitos de 0 a 9)
confusion_matrix = np.zeros((num_classes, num_classes), dtype=int)

# Preencher a matriz acumulando as combinações de classes reais (linhas) e preditas (colunas)
for real, pred in zip(Y_test_raw, test_preds_final):
    confusion_matrix[real, pred] += 1

# Configurar a figura para exibição gráfica da Matriz de Confusão
plt.figure(figsize=(8, 6))
plt.imshow(confusion_matrix, cmap='Blues', interpolation='nearest')
plt.title('Matriz de Confusão - Configuração 1 (ReLU)')
plt.colorbar(label='Quantidade de Classificações')

# Definir marcadores dos eixos de 0 a 9
tick_marks = np.arange(num_classes)
plt.xticks(tick_marks, tick_marks)
plt.yticks(tick_marks, tick_marks)

plt.xlabel('Classe Predita')
plt.ylabel('Classe Real')

# Inserir os valores numéricos dentro de cada quadrante da matriz para facilitar a leitura
limiar_cor = confusion_matrix.max() / 2.0
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, format(confusion_matrix[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if confusion_matrix[i, j] > limiar_cor else "black")

plt.tight_layout()
plt.show()

Com base na análise dos gráficos plotados, foi perceptível que para a problemática de classificação de caracteres, a primeira configuração performou melhor, tanto na baixa da perda quanto na acurácia. 

Conforme meus estudos, além do fato da configuração 1 possuir uma camada oculta a mais, inferi que a pior performance da configuração 2 ocorre em razão da Função de Ativação Sigmoide, haja vista que ela apresenta uma problemática de sumir com o gradiente quando há muitas camadas, já que a derivada máxima da sigmóide é apenas 0.25 e, à medida que a rede tenta aprender, multiplicar repetidamente números menores que 0.25 faz com que o gradiente desapareça rapidamente antes de chegar nas primeiras camadas. Nessa esfera, ao comparar com a ReLu, temos que, para qualquer entrada positiva, a derivada sempre será 1, ou seja, o gradiente passa de forma integral pelas camadas. Paralelamente, vale mencionar que os pesos da Configuração 1 foram inicializados considerando a variância correta para a ReLU (He Initialization / Kaiming), o que também pode ter influenciado na melhor performance.